In [3]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add current directory to path
sys.path.append(os.getcwd())

# Ensure the model exists
try:
    import infection_model_npi as model
except ImportError:
    print("Error: 'infection_model_npi.py' not found. Please ensure the model file is in the directory.")
    sys.exit(1)

# ==========================================
# 1. Setup Parameters & Scenarios
# ==========================================

params = model.Params(
    T=150 * 24,  # 60 days
    N=5000,
    alpha=3,
    initial_infected_normal=2,
    initial_infected_multispreader=1,
    ratios=model.AgentTypeRatio(retired=0.2, adult=0.6, child=0.2),
    schools=4,
    primary_care=20,
    hospitals=1,
    other_working_places=100,
    public_places=50
)

# Define Scenarios (Social Ban is excluded)
scenarios = {
    "Baseline (No NPI)": model.InterventionPolicy(),
    
    "Masks Only": model.InterventionPolicy(
        masks_enabled=True, 
        mask_efficiency=0.48
    ),
    
    "School Closure": model.InterventionPolicy(
        school_closure=True
    ),
    
    "Public Places Closed": model.InterventionPolicy(
        public_places_closed=True
    ),
    
    "WFH (50%)": model.InterventionPolicy(
        wfh_enabled=True, 
        wfh_non_essential_fraction=0.5
    ),
    
    "Hard Lockdown": model.InterventionPolicy(
        school_closure=True,
        public_places_closed=True,
        non_essential_workplaces_closed=True,
        masks_enabled=True
    )
}

intervention_time = 10.0 * 24  # 10th day in hours
NUM_SEEDS = 10
SEEDS = range(42, 42 + NUM_SEEDS)

# ==========================================
# 2. Run Simulations
# ==========================================

all_data = []

print(f"Running simulations for {NUM_SEEDS} seeds...")

for seed in SEEDS:
    print(f"  Processing Seed: {seed}")
    for name, policy in scenarios.items():
        # Init and Run
        sim = model.Simulation(params=params, seed=seed)
        sim.init()
        sim.schedule_intervention(intervention_time, policy)
        sim.run()
        
        # Extract stats
        stats = sim.get_stats_df()
        df = pd.DataFrame(stats)
        
        # Add metadata
        df['Scenario'] = name
        df['Seed'] = seed
        df['Day'] = df['time'] / 24.0
        
        # Calculated columns
        df['Active_Infected'] = df['I0'] + df['I1'] + df['I2'] + df['H']
        df['Total_Affected'] = params.N - df['S']
        df['I_total'] = df['I0'] + df['I1'] + df['I2'] 
        
        all_data.append(df)

# ==========================================
# 3. Store and Read CSV
# ==========================================

full_df = pd.concat(all_data, ignore_index=True)
csv_filename = 'simulation_results_multi_seed.csv'
print(f"Saving results to {csv_filename}...")
full_df.to_csv(csv_filename, index=False)

print("Reading results back for analysis...")
df_analysis = pd.read_csv(csv_filename)

# ==========================================
# 4. Statistical Analysis
# ==========================================

# Group by Scenario and Seed to get peaks for each run
per_run_stats = df_analysis.groupby(['Scenario', 'Seed']).agg({
    'Active_Infected': 'max',
    'H': 'max'
}).reset_index()

# Find peak times
peak_times = []
for (name, seed), group in df_analysis.groupby(['Scenario', 'Seed']):
    idx_max_inf = group['Active_Infected'].idxmax()
    day_max_inf = group.loc[idx_max_inf, 'Day']
    peak_times.append({'Scenario': name, 'Seed': seed, 'Peak_Time_Day': day_max_inf})

df_peak_times = pd.DataFrame(peak_times)
per_run_stats = pd.merge(per_run_stats, df_peak_times, on=['Scenario', 'Seed'])

# Get Baseline stats for comparison
baseline_stats = per_run_stats[per_run_stats['Scenario'] == "Baseline (No NPI)"].set_index('Seed')

results_summary = []

for name in scenarios.keys():
    scenario_stats = per_run_stats[per_run_stats['Scenario'] == name].set_index('Seed')
    
    # Calculate means
    mean_peak_inf = scenario_stats['Active_Infected'].mean()
    mean_peak_hosp = scenario_stats['H'].mean()
    mean_peak_time = scenario_stats['Peak_Time_Day'].mean()
    
    # Calculate improvements per seed relative to that seed's baseline
    inf_reduction = (baseline_stats['Active_Infected'] - scenario_stats['Active_Infected']) / baseline_stats['Active_Infected'] * 100
    hosp_reduction = (baseline_stats['H'] - scenario_stats['H']) / baseline_stats['H'] * 100
    delay = scenario_stats['Peak_Time_Day'] - baseline_stats['Peak_Time_Day']
    
    results_summary.append({
        'Scenario': name,
        'Mean Peak Infected': mean_peak_inf,
        'Inf Reduction (%)': inf_reduction.mean(),
        'Mean Peak Hosp': mean_peak_hosp,
        'Hosp Reduction (%)': hosp_reduction.mean(),
        'Peak Time (Day)': mean_peak_time,
        'Delay (Days)': delay.mean()
    })

df_summary = pd.DataFrame(results_summary)
print("\n=== Simulation Summary (Avg across 10 seeds) ===")
print(df_summary.round(2).to_string())

# ==========================================
# 5. Plotting (Aggregated)
# ==========================================

sns.set_theme(style="whitegrid")

# Aggregate for mean plots
df_agg = df_analysis.groupby(['Scenario', 'Day']).agg({
    'Active_Infected': ['mean', 'std'],
    'Total_Affected': ['mean'],
    'S': 'mean', 'E': 'mean', 'I_total': 'mean', 
    'H': 'mean', 'R': 'mean', 'D': 'mean'
}).reset_index()
df_agg.columns = ['_'.join(col).strip() if col[1] else col[0] for col in df_agg.columns.values]

# --- Aggregated Plot 1: Comparison of Active Infections ---
plt.figure(figsize=(10, 6))
for name in scenarios.keys():
    subset = df_agg[df_agg['Scenario'] == name]
    plt.plot(subset['Day'], subset['Active_Infected_mean'], label=name, linewidth=2)
    plt.fill_between(subset['Day'], 
                     subset['Active_Infected_mean'] - subset['Active_Infected_std'],
                     subset['Active_Infected_mean'] + subset['Active_Infected_std'],
                     alpha=0.1)
plt.axvline(x=10, color='black', linestyle='--', alpha=0.5, label='Intervention')
plt.title(f"Active Infections (Mean ± Std, {NUM_SEEDS} Seeds)", fontsize=12)
plt.xlabel("Days")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig('comparison_active_infections_mean.png', dpi=100)
plt.close()

# --- Aggregated Plot 2: SEIR Grid ---
colors = {"S": "tab:blue", "E": "gold", "I_total": "orange", "H": "tab:red", "R": "tab:green", "D": "black"}
num_scenarios = len(scenarios)
cols = 2
rows = (num_scenarios + 1) // 2
fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
axes_flat = axes.flatten()

for i, name in enumerate(scenarios.keys()):
    ax = axes_flat[i]
    subset = df_agg[df_agg['Scenario'] == name]
    ax.plot(subset['Day'], subset['S_mean'], label="Susceptible", color=colors["S"], linestyle="--", linewidth=1)
    ax.plot(subset['Day'], subset['E_mean'], label="Exposed", color=colors["E"], linewidth=1.5)
    ax.plot(subset['Day'], subset['I_total_mean'], label="Infectious (Total)", color=colors["I_total"], linewidth=2)
    ax.plot(subset['Day'], subset['H_mean'], label="Hospitalized", color=colors["H"], linewidth=1.5)
    ax.plot(subset['Day'], subset['R_mean'], label="Recovered", color=colors["R"], linewidth=1.5)
    ax.plot(subset['Day'], subset['D_mean'], label="Dead", color=colors["D"], linewidth=1.5)
    ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel("Days", fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.grid(True, alpha=0.3)
    if i == 0: ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

for j in range(i + 1, len(axes_flat)): axes_flat[j].axis('off')
plt.tight_layout()
plt.savefig('seir_grid_mean_trajectories.png', dpi=100)
plt.close()

# ==========================================
# 6. Plotting (Individual Seeds)
# ==========================================

print("Generating individual plots for each seed...")
output_folder = "plots_per_seed"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Get unique seeds from the loaded dataframe
unique_seeds = df_analysis['Seed'].unique()

for seed in unique_seeds:
    # Filter data for this seed
    seed_data = df_analysis[df_analysis['Seed'] == seed]
    
    plt.figure(figsize=(10, 6))
    
    # Plot every scenario for this specific seed
    for name in scenarios.keys():
        subset = seed_data[seed_data['Scenario'] == name]
        plt.plot(subset['Day'], subset['Active_Infected'], label=name, linewidth=2)
        
    plt.axvline(x=10, color='gray', linestyle='--', label='Intervention Start')
    plt.title(f"Comparison of Active Infections - Seed {seed}", fontsize=14)
    plt.xlabel("Days")
    plt.ylabel("Count")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save to the dedicated folder
    filename = os.path.join(output_folder, f"comparison_active_infections_seed_{seed}.png")
    plt.savefig(filename, dpi=100)
    plt.close()

print(f"Individual seed plots saved to folder: '{output_folder}'")
print("All tasks complete.")

Running simulations for 10 seeds...
  Processing Seed: 42
  Processing Seed: 43
  Processing Seed: 44
  Processing Seed: 45
  Processing Seed: 46
  Processing Seed: 47
  Processing Seed: 48
  Processing Seed: 49
  Processing Seed: 50
  Processing Seed: 51
Saving results to simulation_results_multi_seed.csv...
Reading results back for analysis...

=== Simulation Summary (Avg across 10 seeds) ===
               Scenario  Mean Peak Infected  Inf Reduction (%)  Mean Peak Hosp  Hosp Reduction (%)  Peak Time (Day)  Delay (Days)
0     Baseline (No NPI)              2310.5               0.00           250.6                0.00            40.76          0.00
1            Masks Only              1157.7              49.86           129.6               48.10            51.10         10.34
2        School Closure              2275.5               1.47           238.1                4.59            46.70          5.94
3  Public Places Closed              2160.9               6.44           231.8    

In [4]:
import os
import matplotlib.pyplot as plt

# 1. Setup Folder and Styles
output_folder = "plots_per_seed"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

colors = {
    "S": "tab:blue", "E": "gold", "I_total": "orange",
    "H": "tab:red", "R": "tab:green", "D": "black"
}

# 2. Loop through every seed
unique_seeds = df_analysis['Seed'].unique()

print(f"Generating SEIR grids for {len(unique_seeds)} seeds...")

for seed in unique_seeds:
    # Filter data for this specific seed
    seed_df = df_analysis[df_analysis['Seed'] == seed]
    
    # Setup Grid
    num_scenarios = len(scenarios)
    cols = 2
    rows = (num_scenarios + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes_flat = axes.flatten()

    # Plot each scenario
    for i, name in enumerate(scenarios.keys()):
        ax = axes_flat[i]
        subset = seed_df[seed_df['Scenario'] == name]
        
        # Plot curves
        ax.plot(subset['Day'], subset['S'], label="Susceptible", color=colors["S"], linestyle="--", linewidth=1)
        ax.plot(subset['Day'], subset['E'], label="Exposed", color=colors["E"], linewidth=1.5)
        ax.plot(subset['Day'], subset['I_total'], label="Infectious (Total)", color=colors["I_total"], linewidth=2)
        ax.plot(subset['Day'], subset['H'], label="Hospitalized", color=colors["H"], linewidth=1.5)
        ax.plot(subset['Day'], subset['R'], label="Recovered", color=colors["R"], linewidth=1.5)
        ax.plot(subset['Day'], subset['D'], label="Dead", color=colors["D"], linewidth=1.5)
        
        # Styling
        ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5)
        ax.set_title(f"{name}", fontsize=11, fontweight='bold')
        ax.set_xlabel("Days", fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Legend only on first plot
        if i == 0:
            ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

    # Hide empty subplots
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis('off')

    plt.suptitle(f"SEIRHD Dynamics - Seed {seed}", fontsize=14, y=0.99)
    plt.tight_layout()
    
    # Save
    filename = os.path.join(output_folder, f"seir_grid_seed_{seed}.png")
    plt.savefig(filename, dpi=100)
    plt.close() # Close memory to prevent RAM issues

print(f"Done! SEIR grids saved in '{output_folder}'.")


Generating SEIR grids for 10 seeds...
Done! SEIR grids saved in 'plots_per_seed'.
